# Kp-Vsys — notebook éducatif

Ce notebook explique chaque étape du pipeline de détection d'atmosphères d'exoplanètes
par spectroscopie haute résolution (HRCCS). L'objectif est qu'un lecteur puisse
décrire les figures dans la section Méthodes d'un article scientifique.

**Contenu**
1. Concept : la spectroscopie haute résolution
2. La vraisemblance logL (prescription Brogi & Line)
3. Prescription Gibson
4. La fonction de cross-corrélation (CCF)
5. Pourquoi marginaliser sur l'amplitude α ?
6. Stabilité numérique et normalisation
7. Méthodes d'intégration comparées
8. Oversampling
9. Lecture de la carte Kp-vsys
10. Le trailing map
11. Contribution par ordre spectral (leave-one-out)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate

import starships.logl_grid as lg
from starships.plotting_fcts import (
    plot_kpvsys_map,
    plot_trailing_map,
    plot_loo_contributions,
    pcolormesh_ts,
)

In [ ]:
# --- Chemins et paramètres (adapter à votre environnement) ---
path_results = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/'
file_glob    = 'take3_HRR_modif_disso_31254924_all_kp*_visit*.npz'
kp_ref       = 227.15  # km/s
rv_expected  = 0.0     # km/s

files = sorted(path_results.glob(file_glob))
lg.load_logl_results(files)

alpha_frac = lg._loaded_extra['alpha_frac']
(idx_signal,) = np.nonzero(alpha_frac > 0.5)
print(f'{len(files)} visite(s), {len(idx_signal)}/{len(alpha_frac)} exposures avec signal')

## 1. Concept : spectroscopie haute résolution (HRCCS)

En spectroscopie haute résolution ($R \gtrsim 25\,000$), les raies moléculaires de
l'atmosphère planétaire sont résolues individuellement. Le mouvement orbital de la
planète (vitesse radiale $v_p$) déplace ces raies en fonction de la phase orbitale :

$$v_p(\phi) = K_p \sin(2\pi\phi) + v_\mathrm{sys}$$

où $K_p$ est la demi-amplitude de la vitesse radiale, $v_\mathrm{sys}$ la vitesse
systémique, et $\phi$ la phase orbitale.

L'idée clef : en balayant une grille $(v_\mathrm{sys}, K_p)$, on cherche la combinaison
qui maximise la cohérence entre les données et le modèle de spectre atmosphérique.
C'est ce que font la CCF et le logL.

## 2. La vraisemblance logL — prescription Brogi & Line

La prescription de Brogi & Line (2019, AJ, 157, 114) maximise la vraisemblance
d'un modèle de spectre $f$ mis à l'échelle par un facteur $\alpha$ :

$$\log \mathcal{L}_\mathrm{BL} = -\frac{N}{2} \log\!\left(s_f^2 - 2\alpha\,\mathrm{CT} + \alpha^2\,\mathrm{ST}\right)$$

où les termes pré-calculés pour chaque nœud de la grille $(v_\mathrm{sys}, K_p)$ sont :

| Terme | Définition | Signification |
|---|---|---|
| $s_f^2$ | $\frac{1}{N}\sum_i \frac{d_i^2}{\sigma_i^2}$ | variance des données |
| $\mathrm{CT}$ | $\frac{1}{N}\sum_i \frac{d_i\,f_i}{\sigma_i^2}$ | corrélation croisée données–modèle |
| $\mathrm{ST}$ | $\frac{1}{N}\sum_i \frac{f_i^2}{\sigma_i^2}$ | auto-corrélation du modèle |

Cette formule découle d'une vraisemblance gaussienne en supposant que le modèle est
correct à un facteur $\alpha$ près. Elle est équivalente à $-N/2\,\log(\chi^2/N)$.
Lorsque $\alpha = 1$, le logL mesure directement l'accord données–modèle.

**Propriété additive** : $\log\mathcal{L}_\mathrm{total} = \sum_i \log\mathcal{L}_i$
sur les exposures et les ordres — ce qui permet de combiner les visites et d'identifier
la contribution de chaque exposition.

In [ ]:
%matplotlib inline

# logL BL à alpha=1, profil 1D au Kp de référence
i_kp = int(np.argmin(np.abs(lg.kp_axis - kp_ref)))
logl_bl = lg.get_logl(idx_exposure=idx_signal, alpha=1., kind='BL',
                       sum_axis=(-2, -1))[:, i_kp]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lg.vsys_axis, logl_bl, 'k', lw=1.2)
ax.axvline(rv_expected, linestyle='--', color='r', alpha=0.6)
ax.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax.set_ylabel(r'$\log\mathcal{L}_{\rm BL}$', fontsize=13)
ax.set_title(f'Profil logL BL à $K_p$ = {lg.kp_axis[i_kp]:.1f} km/s', fontsize=11)
plt.tight_layout()

## 3. Prescription Gibson

Gibson et al. (2020, MNRAS, 493, 2215) proposent un logL qui marginalise
analytiquement sur l'amplitude du bruit $\beta$ :

$$\log \mathcal{L}_\mathrm{G} = -\frac{N}{2} \log s_f^2
  - \frac{N-1}{2} \log\!\left(1 - \rho^2\right)
  + \text{const}$$

où $\rho = \mathrm{CT}/\sqrt{s_f^2 \cdot \mathrm{ST}}$ est le coefficient de
corrélation de Pearson. Contrairement à BL, cette formule ne suppose pas que le
niveau du bruit est parfaitement connu.

En pratique, les deux prescriptions donnent des cartes Kp-vsys cohérentes pour
les détections fortes, mais Gibson peut être plus robuste en présence de résidus
systématiques dans le spectre.

In [ ]:
# Comparaison BL vs Gibson sur le profil 1D
logl_g = lg.get_logl(idx_exposure=idx_signal, alpha=1., kind='G',
                      sum_axis=(-2, -1))[:, i_kp]

# Normaliser chaque profil à sa médiane hors-fenêtre pour comparer les formes
noise_rv = 15.  # km/s
is_out = np.abs(lg.vsys_axis - rv_expected) > noise_rv

def _norm(x):
    x = x - np.ma.median(x[is_out])
    return x / np.ma.std(x[is_out])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lg.vsys_axis, _norm(logl_bl), color='#0072B2', lw=1.5, label='Brogi & Line')
ax.plot(lg.vsys_axis, _norm(logl_g),  color='#D55E00', lw=1.5, linestyle='--', label='Gibson')
ax.axvline(rv_expected, linestyle=':', color='gray', lw=0.8)
ax.axhline(0, linestyle=':', color='gray', lw=0.8)
ax.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax.set_ylabel('S/N', fontsize=13)
ax.legend(fontsize=11)
ax.set_title('Comparaison BL vs Gibson (profil 1D normalisé)', fontsize=11)
plt.tight_layout()

## 4. La fonction de cross-corrélation (CCF)

La CCF est un cas particulier du logL lorsqu'on ne tient pas compte des incertitudes
individuelles ($\sigma_i = \text{const}$) et qu'on omet les termes d'auto-corrélation :

$$\mathrm{CCF}(v) = \sum_i d_i \cdot f_i(v)$$

Elle est plus simple à interpréter qualitativement (unités de flux × flux) mais
moins rigoureuse statistiquement. Dans STARSHIPS, la CCF est calculée en utilisant
le terme CT normalisé :

$$\mathrm{CCF}_\mathrm{BL}(v) = N \cdot \mathrm{CT}(v)$$

In [ ]:
# CCF 1D au Kp de référence
ccf_1d = lg.get_ccf(idx_exposure=idx_signal, kind='BL',
                     sum_axis=(-2, -1))[:, i_kp]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3))

ax1.plot(lg.vsys_axis, _norm(logl_bl), color='#0072B2', lw=1.5, label='logL BL')
ax1.plot(lg.vsys_axis, _norm(ccf_1d),  color='#E69F00', lw=1.5, linestyle='--', label='CCF BL')
ax1.axvline(rv_expected, linestyle=':', color='gray', lw=0.8)
ax1.axhline(0, linestyle=':', color='gray', lw=0.8)
ax1.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax1.set_ylabel('S/N', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_title('logL vs CCF (profil 1D)', fontsize=11)

# Corrélation entre logL et CCF
ax2.scatter(_norm(ccf_1d), _norm(logl_bl), s=8, alpha=0.6, color='#0072B2')
ax2.set_xlabel('CCF normalisé', fontsize=13)
ax2.set_ylabel('logL normalisé', fontsize=13)
ax2.set_title('Corrélation logL–CCF', fontsize=11)
plt.tight_layout()

## 5. Pourquoi marginaliser sur l'amplitude α ?

Le facteur $\alpha$ représente l'amplitude réelle du modèle de spectre atmosphérique.
$\alpha = 1$ signifie que le modèle est parfaitement calibré en amplitude ;
$\alpha < 1$ indique que le signal observé est plus faible que prédit (modèle trop riche
en absorptions, ou planète partiellement occultée) ; $\alpha > 1$ l'inverse.

Plutôt que de maximiser sur $\alpha$ (ce qui introduirait un biais), on marginalise :

$$P(K_p, v_\mathrm{sys}\,|\,\mathbf{d}) =
  \int_0^{\alpha_\max} \mathcal{L}(\alpha, K_p, v_\mathrm{sys})\, p(\alpha)\, \mathrm{d}\alpha$$

On utilise un prior uniforme $p(\alpha) = \text{const}$ sur $[0.01, 2]$ —
couvrant les facteurs d'amplitude sub- à supra-solaire.

**Avantage** : la carte marginalisée est moins sensible aux imperfections du modèle
atmosphérique. Elle permet aussi de contraindre $\alpha$ directement (section 6 de
`kpvsys_products.ipynb`).

In [ ]:
# Effet de la valeur de alpha sur le profil logL
alpha_vals = [0.5, 1.0, 1.5, 2.0]
_COLS = ['#66CCEE', '#0072B2', '#4477AA', '#332288']

fig, ax = plt.subplots(figsize=(8, 3.5))
for alpha, col in zip(alpha_vals, _COLS):
    l = lg.get_logl(idx_exposure=idx_signal, alpha=alpha, kind='BL',
                    sum_axis=(-2, -1))[:, i_kp]
    ax.plot(lg.vsys_axis, _norm(l), color=col, lw=1.4, label=f'α = {alpha}')
ax.axvline(rv_expected, linestyle=':', color='gray', lw=0.8)
ax.axhline(0, linestyle=':', color='gray', lw=0.8)
ax.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax.set_ylabel('S/N', fontsize=13)
ax.legend(fontsize=10, ncol=2)
ax.set_title('Profil logL BL pour différentes valeurs de α', fontsize=11)
plt.tight_layout()

## 6. Stabilité numérique — pré-normalisation

Le logL BL est $-N/2 \times \log(\chi^2/N)$. Pour un jeu de données de grande taille
($N \sim 10^5$–$10^6$ pixels), le logL peut atteindre des valeurs très négatives
(p. ex. $-73\,000$). Lors de la marginalisation, on doit calculer
$\int \exp(\log\mathcal{L})\,\mathrm{d}\alpha$ — mais $\exp(-73\,000) = 0$ en
virgule flottante 64-bit, ce qui donne un posterior nul partout.

**Solution** : soustraire le maximum global du cube logL avant tout appel à `exp()` :

$$\log\mathcal{L}^* = \log\mathcal{L} - \max_{\alpha,v,K}\log\mathcal{L}$$

Les valeurs passent de $(-73\,480, -73\,026)$ à $(-454, 0)$, restant bien dans la
plage de `float64`. Cette normalisation est purement numérique : elle annule
identiquement dans le rapport de vraisemblance et n'affecte pas le posterior final.

In [ ]:
alpha_array = np.linspace(0.01, 2., 31)

# Cube brut — vectorisé sur alpha (un seul appel NumPy)
logl_raw = lg.get_logl(
    idx_exposure=idx_signal, alpha=alpha_array, kind='BL', sum_axis=(-2, -1),
)  # shape (n_alpha, n_vsys, n_kp)
print(f'Range brut  : ({logl_raw.min():.2f}, {logl_raw.max():.2f})')
print(f'exp(min)    : {np.exp(logl_raw.min())}')

# Après pré-normalisation (comme dans _build_logl_cube)
finite = logl_raw[np.isfinite(logl_raw)]
logl_norm = logl_raw - float(finite.max())
print(f'Range normalisé : ({logl_norm.min():.2f}, {logl_norm.max():.2f})')
print(f'exp(min normalisé) : {np.exp(logl_norm.min()):.2e}')

## 7. Méthodes d'intégration comparées

La marginalisation est une intégrale 1D sur $\alpha$. Deux méthodes :

- **Somme rectangulaire** : $\sum_i \exp(\log\mathcal{L}(\alpha_i)) \cdot \Delta\alpha$.
  Simple mais peu précise pour des fonctions peu régulières.
- **Règle de Simpson en espace log** (`_log_simps`) : calcule
  $\log\int \exp(\log\mathcal{L})\,\mathrm{d}\alpha$ via l'intégration de Simpson
  de $\exp(\log\mathcal{L} - \text{max})$, puis rajoute le max.
  Numériquement stable et plus précis à même nombre de points d'évaluation.

Les deux méthodes donnent des contours identiques lorsque l'intégrale est bien
échantillonnée (31 points sur $[0.01, 2]$ est largement suffisant pour un logL
lisse en $\alpha$).

In [ ]:
d_alpha = alpha_array[1] - alpha_array[0]

# Méthode 1 : Simpson en espace log (choisi dans le code)
marg_simps = np.exp(lg._log_simps(logl_norm, d_alpha, axis=0))

# Méthode 2 : somme rectangulaire
exp_shift = float(logl_norm[np.isfinite(logl_norm)].max())
marg_rect = np.sum(np.exp(logl_norm - exp_shift), axis=0) * d_alpha * np.exp(exp_shift)

# Comparaison sur une tranche vsys au Kp de référence
fig, ax = plt.subplots(figsize=(8, 3))
with np.errstate(divide='ignore'):
    ax.plot(lg.vsys_axis, np.log(marg_simps[:, i_kp]), color='#0072B2', lw=1.5, label='Simpson log-espace')
    ax.plot(lg.vsys_axis, np.log(marg_rect[:, i_kp]),  color='#D55E00', lw=1.5, ls='--', label='Somme rectangulaire')
ax.set_xlabel(r'$v_\mathrm{sys}$ (km s$^{-1}$)', fontsize=13)
ax.set_ylabel(r'$\log P(K_p, v_\mathrm{sys}\,|\,\mathbf{d})$', fontsize=13)
ax.set_title(f'Comparaison méthodes — Kp = {lg.kp_axis[i_kp]:.1f} km/s', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()

## 8. Oversampling

La grille $(v_\mathrm{sys}, K_p)$ calculée sur le cluster a un pas fixe, typiquement
1–2 km/s. Pour déterminer les contours sigma avec précision, `get_contours_posterior`
trie la grille et accumule la masse de probabilité — sur une grille grossière, chaque
pas de la somme cumulative représente une large fraction de la masse totale, ce qui
donne des niveaux sigma peu précis.

L'oversampling par spline cubique (en espace log pour éviter les oscillations)
multiplie la résolution sans recalculer la grille logL coûteuse. Un facteur
d'oversampling de 2 est un bon compromis précision/mémoire.

**Remarque** : l'oversampling se fait *après* la marginalisation sur $\alpha$,
ce qui est correct car les axes $(v_\mathrm{sys}, K_p)$ et $\alpha$ sont indépendants
dans l'intégrale.

In [ ]:
# Comparaison oversampling 1 vs 2 sur les niveaux de contours sigma
from starships.plotting_fcts import get_contours_posterior, sigma2percent_2d

for os_factor, label in [(1, 'sans oversampling'), (2, 'oversampling ×2')]:
    post, vsys_os, kp_os, *_ = lg.compute_kpvsys_posterior(
        alpha_array=alpha_array, idx_signal=idx_signal, oversample=os_factor,
    )
    d_v = vsys_os[1] - vsys_os[0]
    d_k = kp_os[1] - kp_os[0]
    lvl, _ = get_contours_posterior(post, [d_v, d_k], lvls=[0.3935, 0.8647], renormalize=True)
    print(f'{label} (pas={d_v:.2f}×{d_k:.2f} km/s) — niveaux 1σ,2σ : {lvl}')

## 9. Lecture de la carte Kp-vsys

La carte Kp-vsys est le produit central de l'analyse HRCCS. Elle montre la
probabilité marginalisée $P(K_p, v_\mathrm{sys}\,|\,\mathbf{d})$ sous forme de carte de
chaleur, avec des contours à 1, 2, 3σ.

**Ce que les axes signifient :**
- **Axe x ($v_\mathrm{sys}$)** : vitesse systémique du système planétaire. Sa valeur
  attendue est la vitesse radiale de l'étoile hôte (souvent connue des éphémérides).
- **Axe y ($K_p$)** : demi-amplitude de la vitesse radiale orbitale. Elle se calcule
  à partir de la période $P$, de la masse stellaire $M_*$ et du demi-grand axe :
  $K_p = 2\pi a \sin i / P$.

**Interpréter les contours :**
- Un pic à $(v_\mathrm{sys,\,ref}, K_{p,\,\rm ref})$ confirme la détection.
- Un pic déplacé en $v_\mathrm{sys}$ indique une erreur sur les éphémérides ou
  un vent planétaire.
- Un pic déplacé en $K_p$ peut signaler que l'orbite circulaire supposée est
  inexacte (excentricité non nulle).
- Un pic large traduit un faible signal (S/N bas).
- Plusieurs pics : le modèle correspond à plusieurs configurations ou il y a
  un artefact systématique.

In [ ]:
posterior, vsys_os, kp_os, margin_vsys, margin_kp = lg.compute_kpvsys_posterior(
    alpha_array=alpha_array, idx_signal=idx_signal, oversample=2,
)

fig, axes = plot_kpvsys_map(
    posterior, vsys_os, kp_os, margin_vsys, margin_kp,
    n_sigma=3, sigma_display='contours',
)
ax_map = axes[0]
ax_map.axvline(rv_expected, color='r', linestyle='--', alpha=0.5, label='vsys attendu')
ax_map.axhline(kp_ref,      color='r', linestyle='--', alpha=0.5, label='Kp attendu')
ax_map.legend(fontsize=10)

## 10. Le trailing map

Le trailing map (ou trailing plot) est une représentation complémentaire qui montre
la cohérence du signal en fonction du temps (phase orbitale).

**Comment le construire :**
Pour un $K_p$ fixé, on calcule le logL normalisé pour chaque exposition individuellement,
en fonction de la vitesse systémique supposée $v_\mathrm{sys}$ :

$$\mathcal{M}(v_\mathrm{sys}, \phi_i) =
  \frac{\log\mathcal{L}_i(v_\mathrm{sys}) - \mu_i^\mathrm{bruit}}{\sigma_i^\mathrm{bruit}}$$

où $\mu_i$ et $\sigma_i$ sont estimés dans la fenêtre de bruit hors-pic.

**Ce qu'on voit :**
- Un signal planétaire apparaît comme une **strie inclinée** : à chaque phase, la
  vitesse radiale de la planète change, déplaçant le pic en $v_\mathrm{sys}$ selon
  $\Delta v = K_p \sin(2\pi\phi)$.
- Le panneau de droite montre la **lightcurve du pic** : la moyenne du logL normalisé
  dans la fenêtre $v_\mathrm{sys}$ de signal. On y superpose la **contribution
  fractionnelle** $\Delta\mathcal{L}_i / \mathcal{L}_\mathrm{comb}$ (en vermillon) :
  les exposures qui contribuent le plus au signal combiné devraient être celles où
  la planète est visible ($\alpha_\mathrm{frac} > 0.5$).
- Le panneau du bas est le profil logL 1D intégré sur toutes les exposures de signal.

In [ ]:
# Trailing map pour la première visite (exemple)
noise_rv_width = 15.   # km/s
peak_rv_width  =  2.   # km/s
noise_rv_limits = (rv_expected - noise_rv_width, rv_expected + noise_rv_width)
peak_rv_limits  = (rv_expected - peak_rv_width,  rv_expected + peak_rv_width)
is_out_rv = (lg.vsys_axis < noise_rv_limits[0]) | (noise_rv_limits[1] < lg.vsys_axis)

# Profil 1D combiné (toutes visites, pour le panneau du bas)
logl_1d_all = lg.get_logl(idx_exposure=idx_signal, alpha=1., sum_axis=(-2, -1))[:, i_kp]
logl_1d_all_norm = logl_1d_all - np.ma.median(logl_1d_all[is_out_rv])
logl_1d_all_norm /= np.ma.std(logl_1d_all_norm[is_out_rv])

# Trailing map de la première visite
lg.load_logl_results([files[0]])
phase = lg._loaded_extra.get('phase', np.arange(lg.N.shape[-2]))
(visit_signal,) = np.nonzero(lg._loaded_extra['alpha_frac'] > 0.5)

logl_map_ts = lg.get_logl(alpha=1., sum_axis=-1)[:, i_kp, :]  # (n_vsys, n_exp)
logl_map_norm = (logl_map_ts - np.ma.median(logl_map_ts[is_out_rv, :], axis=0)[None, :])
logl_map_norm /= np.ma.std(logl_map_ts[is_out_rv, :], axis=0)[None, :]

logl_1d = lg.get_logl(idx_exposure=visit_signal, alpha=1., sum_axis=(-2, -1))[:, i_kp]
logl_1d_norm = logl_1d - np.ma.median(logl_1d[is_out_rv])
logl_1d_norm /= np.ma.std(logl_1d_norm[is_out_rv])

# Contribution fractionnelle au vsys attendu
i_vsys_peak = int(np.argmin(np.abs(lg.vsys_axis - rv_expected)))
delta = logl_map_ts[i_vsys_peak, :] - np.ma.median(logl_map_ts[is_out_rv, :], axis=0)
logl_comb_val = float(np.sum(delta[visit_signal]))
contrib = delta / logl_comb_val if logl_comb_val != 0 else None

contact_phases = lg._loaded_extra.get('contact_phases', None)
phase_contacts = None
if contact_phases is not None and len(contact_phases) == 4:
    phase_contacts = {
        '1_4': [float(contact_phases[0]), float(contact_phases[3])],
        '2_3': [float(contact_phases[1]), float(contact_phases[2])],
    }

fig, axes = plot_trailing_map(
    logl_map_norm, lg.vsys_axis, phase, logl_1d_norm,
    noise_rv_limits=noise_rv_limits,
    peak_rv_limits=peak_rv_limits,
    logl_1d_norm_all=logl_1d_all_norm,
    rv_expected=rv_expected,
    phase_contacts=phase_contacts,
    contrib=contrib,
)
fig.suptitle(f'Trailing map — Visite 1 (Kp = {lg.kp_axis[i_kp]:.1f} km/s)', y=1.02)

lg.load_logl_results(files)  # recharger toutes les visites

### Interpréter le trailing map dans un article

> *"Le trailing map (Fig. X) montre le logL normalisé en fonction de la phase
> orbitale et de la vitesse systémique supposée, à $K_p = K_{p,\rm ref}$.
> Le signal planétaire se manifeste par une strie inclinée dont la pente est
> donnée par $\Delta v / \Delta\phi = K_p \times 2\pi$. La lightcurve du pic
> (panneau droit, points bleus) montre la valeur médiane du logL normalisé dans
> la fenêtre $v_\mathrm{sys} \in [v_0 \pm \Delta v_\mathrm{pic}]$, à comparer
> à la contribution fractionnelle $\Delta\mathcal{L}_i/\mathcal{L}_\mathrm{comb}$
> (points vermillon), qui indique quelle fraction du signal combiné est apportée
> par chaque exposition."*

## 11. Contribution par ordre spectral (leave-one-out)

### 11a. Motivation

La carte Kp-vsys combine tous les ordres spectraux simultanément. On voudrait savoir
**quelle fraction du signal de détection provient de chaque ordre** — autrement dit,
quels intervalles de longueur d'onde portent l'empreinte atmosphérique de la planète.

Cette information est utile pour :
- Confirmer que le signal est cohérent entre plusieurs ordres (robustesse)
- Identifier les ordres contaminés (téllurique, artefact instrumental)
- Relier le signal à des molécules spécifiques (H₂O absorbe à ~1.4 µm, CO à ~2.3 µm…)
- Comparer deux modèles atmosphériques sur leur capacité à reproduire le signal ordre par ordre

### 11b. Pourquoi on ne peut pas simplement sommer les logL par ordre

Il serait tentant d'écrire

$$\log\mathcal{L}_\mathrm{total} \stackrel{?}{=} \sum_k \log\mathcal{L}_k$$

et d'attribuer la fraction $\log\mathcal{L}_k / \log\mathcal{L}_\mathrm{total}$ à l'ordre $k$.
**Ce n'est pas valide** pour la prescription Brogi & Line :

$$\log\mathcal{L}_\mathrm{BL} = -\frac{N}{2}\log\frac{\chi^2}{N}$$

Le facteur $N/2$ couplé au logarithme fait que la logL totale n'est PAS la somme des
logL par ordre. Seuls les **composantes chi²** sont additives :

$$\chi^2_\mathrm{total} = \sum_k \chi^2_k \qquad
  s_f^2_\mathrm{total} = \sum_k s_{f,k}^2 \qquad
  \mathrm{CT}_\mathrm{total} = \sum_k \mathrm{CT}_k \qquad
  \mathrm{ST}_\mathrm{total} = \sum_k \mathrm{ST}_k$$

C'est ce que `get_chi2_components` exploite : pour calculer la logL *sans* l'ordre $k$,
on soustrait simplement ses composantes des totaux, sans refaire les calculs coûteux
sur le cluster.

### 11c. L'approche leave-one-out (LOO)

Pour chaque ordre $k$, on calcule le **posterior Kp-vsys sans cet ordre** :

$$\log\mathcal{L}_{-k}(\alpha, v, K_p) =
  \text{BL}(s_f^2 - s_{f,k}^2,\; \mathrm{CT} - \mathrm{CT}_k,\; \mathrm{ST} - \mathrm{ST}_k,\; N - N_k)$$

puis on marginalise sur $\alpha$ exactement comme pour le posterior complet. Le
posterior LOO s'obtient **sans un seul appel supplémentaire au cluster** — seulement
deux appels à `get_chi2_components` (total + par ordre), suivis d'une boucle légère.

### 11d. Quelle métrique pour comparer full vs LOO ?

Plusieurs approches naïves posent problème :

| Approche | Problème |
|---|---|
| Différence de logL au pic | Le pic peut être spurieux (bruit) |
| $\log Z_\mathrm{full} - \log Z_{-k}$ (intégrale 3D) | Biais par plancher de bruit : les ordres avec beaucoup de pixels et $\chi^2/N$ légèrement $> 1$ hors-signal paraissent tous négatifs |
| Évaluation en un seul point $(v^*, K_p^*)$ | Sensible à l'erreur sur la position du pic |

**La solution retenue : la fraction de probabilité hors-pic.**

On définit la fraction de probabilité *en dehors* de la région du signal :

$$f_\mathrm{off} = \frac{\displaystyle\sum_{(v,K) \in \Omega_\mathrm{off}}
P(v, K)}{\displaystyle\sum_\mathrm{tout} P(v, K)}$$

où $\Omega_\mathrm{off}$ est la zone de la grille simultanément éloignée du signal
**en vsys ET en Kp** (coins de la grille, contrôlés par `vsys_excl` et `kp_excl`).

La **contribution de l'ordre $k$** est :

$$\boxed{\text{contribution}_k = f_\mathrm{off}^{\text{LOO}_k} - f_\mathrm{off}^\mathrm{full}}$$

### 11e. Pourquoi cette métrique fonctionne

**Un bon ordre** concentre la probabilité dans le pic. Le retirer fait *monter* $f_\mathrm{off}$
(la probabilité se disperse). Contribution $> 0$.

**Un mauvais ordre** (contamination tellurique, artefact) étale la probabilité hors-pic.
Le retirer fait *baisser* $f_\mathrm{off}$ (le pic se resserre). Contribution $< 0$.

**Un ordre neutre** n'affecte pas la forme relative du posterior.
$f_\mathrm{off}^\mathrm{LOO} \approx f_\mathrm{off}^\mathrm{full}$. Contribution $\approx 0$.

**Pourquoi c'est robuste au scaling en N :**
On compare des *fractions* — les facteurs $\Delta v \cdot \Delta K_p$ et les effets
d'échelle absolue de la logL (proportionnels à $N$) s'annulent exactement dans le rapport.
Un ordre neutre qui "dilue" uniformément le posterior laisse la fraction $f_\mathrm{off}$
inchangée.

**Pourquoi les "coins" de la grille :**
On utilise une condition AND (loin en vsys ET loin en Kp) pour s'assurer que la région
de mesure est clairement hors de toute signature planétaire plausible. Il y a naturellement
beaucoup plus de points dans cette région que dans le pic, ce qui stabilise l'estimation.

### 11f. Visualisation : barchart + cartes Kp-vsys

La figure LOO comporte deux panneaux :

- **Haut — barchart** : pour chaque ordre, la fraction normalisée
  $\text{contribution}_k / \sum_j \max(\text{contribution}_j, 0)$ (somme des barres bleues = 1).
  Les barres oranges indiquent les ordres néfastes.

- **Bas — cartes Kp-vsys** : pour chaque ordre, la carte
  $\log P_\mathrm{full,norm} - \log P_\mathrm{LOO_k,norm}$
  (chaque posterior normalisé indépendamment, max = 0). Bleu = l'ordre booste le
  posterior à ce point ; rouge = il le détériore.
  Ces cartes révèlent *où* dans l'espace $(v, K_p)$ chaque ordre est actif — utile
  pour identifier des artefacts localisés.

> **Paramètre clef à ajuster :** `vsys_excl` et `kp_excl` (défaut 30 km/s).
> Si la grille est petite ou le signal très large, réduire ces valeurs.
> Un avertissement s'affiche si moins de 10 points hors-pic sont disponibles.

In [ ]:
# ── Compute LOO contributions ────────────────────────────────────────────────
alpha_array = np.linspace(0.01, 2., 31)

# Only two chi² component calls needed — the loop subtracts per-order components
# from the total without recomputing the expensive grid on the cluster.
posterior_loo, log_delta, contributions, contributions_frac, idx_ord_used = \
    lg.compute_loo_order_contributions(
        alpha_array=alpha_array,
        idx_signal=idx_signal,
        kind='BL',
        # kp_ref=kp_ref,       # center of the exclusion zone; defaults to map max
        # vsys_ref=rv_expected,
        vsys_excl=30.,         # half-width of off-peak exclusion in vsys (km/s)
        kp_excl=30.,           # half-width of off-peak exclusion in Kp (km/s)
    )

# contributions[k]      : f_off_loo_k - f_off_full  (positive = good, negative = bad)
# contributions_frac[k] : fraction of total positive signal contributed by order k
best_k  = np.argmax(contributions_frac)
worst_k = np.argmin(contributions_frac)
print(f'{len(contributions)} ordres analysés')
print(f'Ordre le plus contributif : {idx_ord_used[best_k]}  '
      f'({contributions_frac[best_k]*100:.1f} % du signal)')
print(f'Ordre le plus nuisible    : {idx_ord_used[worst_k]}  '
      f'({contributions_frac[worst_k]*100:.1f} % du signal)')

# Show how many orders are positive / negative / null
n_pos  = int((contributions > 0).sum())
n_neg  = int((contributions < 0).sum())
n_zero = int((contributions == 0).sum())
print(f'\nOrdres positifs (signal)  : {n_pos}')
print(f'Ordres négatifs (artefact): {n_neg}')
print(f'Ordres vides / nuls       : {n_zero}')

In [ ]:
%matplotlib inline

# ── Figure LOO ───────────────────────────────────────────────────────────────
fig, (ax_bar, axes_maps) = plot_loo_contributions(
    posterior_loo, log_delta, contributions,
    vsys_axis=lg.vsys_axis,
    kp_axis=lg.kp_axis,
    idx_orders=idx_ord_used,
    contributions_frac=contributions_frac,   # show fraction instead of raw Δf
    n_col=6,
    # kp_ref=kp_ref,                         # mark expected signal location on maps
    # vsys_ref=rv_expected,
    # vsys_lim=(-50, 50),                    # zoom on the detection region
    # kp_lim=(150, 300),
)

# ── Diagnostic: visualise the off-peak mask ──────────────────────────────────
vsys_excl = 30.  # must match what was passed to compute_loo_order_contributions
kp_excl   = 30.

# Locate the reference point (map maximum, since kp_ref/vsys_ref not set here)
peak_ind = np.unravel_index(np.argmax(posterior_loo), posterior_loo.shape)
vsys_ctr = lg.vsys_axis[peak_ind[0]]
kp_ctr   = lg.kp_axis[peak_ind[1]]

off_vsys = np.abs(lg.vsys_axis - vsys_ctr) > vsys_excl
off_kp   = np.abs(lg.kp_axis   - kp_ctr)  > kp_excl
off_mask = np.outer(off_vsys, off_kp)

print(f'Centre de la zone d\'exclusion : vsys = {vsys_ctr:.1f} km/s, Kp = {kp_ctr:.1f} km/s')
print(f'Points hors-pic disponibles : {off_mask.sum()} / {off_mask.size} '
      f'({100*off_mask.mean():.0f} % de la grille)')

# Overlay the off-peak mask on the full posterior
fig2, ax2 = plt.subplots(figsize=(6, 5))
ax2.pcolormesh(lg.vsys_axis, lg.kp_axis, posterior_loo.T, cmap='Blues')
# Shade the excluded (on-peak) region
on_mask = ~off_mask
ax2.contourf(lg.vsys_axis, lg.kp_axis, on_mask.T.astype(float),
             levels=[0.5, 1.5], colors=['orange'], alpha=0.35)
ax2.axvline(vsys_ctr, color='r', lw=0.8, ls='--')
ax2.axhline(kp_ctr,   color='r', lw=0.8, ls='--')
ax2.set_xlabel(r'$v_\mathrm{sys}$ (km/s)', fontsize=12)
ax2.set_ylabel(r'$K_p$ (km/s)', fontsize=12)
ax2.set_title('Posterior + zone d\'exclusion (orange = exclue du calcul hors-pic)',
              fontsize=10)
plt.tight_layout()

### 11g. Interpréter les résultats

**Barchart — que regarder :**
- La plupart des barres bleues devraient être de petite amplitude et à peu près égales
  (ordres qui portent un peu du signal).
- Quelques ordres devraient se démarquer nettement (ceux qui contiennent les raies
  moléculaires les plus nombreuses ou les plus profondes).
- Les barres oranges signalent des ordres à examiner : contamination tellurique
  résiduelle, correction de la vitesse terrestre incorrecte, ou raies stellaires
  mal soustraites.

**Cartes Kp-vsys par ordre :**
- Une carte essentiellement bleue autour de $(v^*, K_p^*)$ confirme que l'ordre
  contribue au pic de détection.
- Une carte rouge au centre signifie que l'ordre *détériore* le signal à cet endroit
  (il ajoute de la probabilité ailleurs).
- Une carte uniforme (ni bleu ni rouge marqués) indique un ordre neutre/vide.

**Caveats :**

1. **Zone d'exclusion trop petite** : si `vsys_excl` ou `kp_excl` est inférieur à la
   largeur typique du pic (~5–15 km/s), la zone hors-pic contiendra les ailes du pic,
   biaisant la métrique vers le bas. Augmenter `vsys_excl` et `kp_excl`.

2. **Zone d'exclusion trop grande** : si la grille est petite, les coins peuvent avoir
   moins de 10 points. Réduire `vsys_excl` et `kp_excl`, ou choisir `kp_ref`/`vsys_ref`
   manuellement pour centrer la zone d'exclusion sur le signal attendu plutôt que sur
   le maximum de la carte.

3. **Détection marginale** : pour un signal faible ($< 3\sigma$), le posterior est
   diffus et les contributions individuelles seront petites et bruitées. La métrique
   reste valide mais interprétée avec prudence.

### 11h. Comment le citer dans un article

> *"Pour évaluer la contribution de chaque ordre spectral à la détection, nous avons
> utilisé une approche leave-one-out (LOO) : pour chaque ordre $k$, nous avons
> calculé le posterior Kp-vsys en retirant les composantes chi² de cet ordre des
> totaux, sans recalculer la grille logL sur le cluster. La contribution est mesurée
> par la variation de la fraction de probabilité dans les coins de la grille
> (|$v_\mathrm{sys} - v^*$| > 30 km/s ET |$K_p - K_p^*$| > 30 km/s) lors du retrait
> de l'ordre k. Cette métrique est insensible aux effets de normalisation absolue de
> la logL et ne requiert pas de connaître la position exacte du pic de détection."*